In [2]:
import re
import os
import sys
import random
import numpy as np
import pandas as pd

path = os.getcwd().replace("\\notebooks", "")
sys.path.append(path)

from src.preprocessing import preprocessing

from nltk.stem.snowball import RussianStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    pairwise_distances,
)


In [3]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

df = preprocessing(path + "\\data\\datasets.csv")
neg_df = df[df["sentiment"] == 2].copy().reset_index(drop=True)

print(f"Всего негативных отзывов: {len(neg_df):,}")
neg_df.head(3)


Всего негативных отзывов: 55,310


,n,text,sentiment,clean_review
0,13019,Не было. Это пиздеш ТСа.\r\n,2,не было это пиздеш тса
1,26422,Очень вкусно готовят и цены приемлемые ! Отзы...,2,очень вкусно готовят и цены приемлемые отзывчи...
2,22919,До качественной медицины нам далеко\r\nПробова...,2,до качественной медицины нам далеко пробовала ...


In [4]:
with open(path + "\\data\\stopswords.txt", "r", encoding="utf-8") as f:
    stopwords = [line.strip() for line in f if line.strip()]

stemmer = RussianStemmer()
TOKEN_RE = re.compile(r"[A-Za-zА-Яа-яЁё]+")

# Стеммируем стоп-слова тоже, чтобы удаление было согласованным
stopwords_stemmed = {stemmer.stem(word.lower()) for word in stopwords}

def normalize_text(text: str):
    if not isinstance(text, str):
        return []
    tokens = TOKEN_RE.findall(text.lower())
    stems = []
    for token in tokens:
        stem = stemmer.stem(token)
        if len(stem) < 2:
            continue
        if stem in stopwords_stemmed:
            continue
        stems.append(stem)
    return stems

test_words = ["врачи", "врач", "врача", "врачей", "лечить", "лечит", "лечения"]
[(w, stemmer.stem(w)) for w in test_words]


[('врачи', 'врач'),
 ('врач', 'врач'),
 ('врача', 'врач'),
 ('врачей', 'врач'),
 ('лечить', 'леч'),
 ('лечит', 'леч'),
 ('лечения', 'лечен')]

In [5]:
vectorizer = TfidfVectorizer(
    tokenizer=normalize_text,
    token_pattern=None,      # обязательно, если используется кастомный tokenizer
    lowercase=False,         # мы уже приводим к нижнему регистру внутри normalize_text
    max_features=20000,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2),
    sublinear_tf=True,
)

X = vectorizer.fit_transform(neg_df["clean_review"].fillna(""))
print("TF-IDF shape:", X.shape)


TF-IDF shape: (55310, 20000)


In [6]:
# Для быстрого и более устойчивого подбора k уменьшаем размерность
svd = TruncatedSVD(n_components=min(100, X.shape[1] - 1), random_state=RANDOM_STATE)
normalizer = Normalizer(copy=False)
X_reduced = make_pipeline(svd, normalizer).fit_transform(X)

print("Reduced shape:", X_reduced.shape)
print("Explained variance ratio:", round(float(svd.explained_variance_ratio_.sum()), 4))


Reduced shape: (55310, 100)
Explained variance ratio: 0.1037


In [7]:
def evaluate_k_range(X_sparse, X_dense, candidate_k, sample_size=3000, random_state=42):
    '''
    X_sparse - исходный TF-IDF (для финальной модели не нужен, но оставлен для гибкости)
    X_dense  - SVD-представление для подбора k
    '''
    n = X_dense.shape[0]
    sample_size = min(sample_size, n)
    sample_idx = np.random.RandomState(random_state).choice(n, size=sample_size, replace=False)
    X_sample = X_dense[sample_idx]

    rows = []
    for k in candidate_k:
        model = MiniBatchKMeans(
            n_clusters=k,
            random_state=random_state,
            n_init=10,
            batch_size=1024,
        )
        labels = model.fit_predict(X_dense)
        sample_labels = labels[sample_idx]

        sil = silhouette_score(X_sample, sample_labels, metric="euclidean")
        ch = calinski_harabasz_score(X_dense, labels)
        db = davies_bouldin_score(X_dense, labels)

        rows.append(
            {
                "k": k,
                "silhouette": sil,
                "calinski_harabasz": ch,
                "davies_bouldin": db,
                "inertia": model.inertia_,
            }
        )
        print(
            f"k={k:>2} | silhouette={sil:.4f} | "
            f"calinski_harabasz={ch:,.0f} | davies_bouldin={db:.4f}"
        )

    metrics_df = pd.DataFrame(rows)

    # Ранжируем сразу по нескольким метрикам:
    # silhouette и calinski_harabasz - больше лучше
    # davies_bouldin - меньше лучше
    metrics_df["rank_silhouette"] = metrics_df["silhouette"].rank(ascending=False, method="min")
    metrics_df["rank_ch"] = metrics_df["calinski_harabasz"].rank(ascending=False, method="min")
    metrics_df["rank_db"] = metrics_df["davies_bouldin"].rank(ascending=True, method="min")
    metrics_df["rank_sum"] = (
        metrics_df["rank_silhouette"]
        + metrics_df["rank_ch"]
        + metrics_df["rank_db"]
    )

    return metrics_df.sort_values(["rank_sum", "silhouette", "calinski_harabasz"], ascending=[True, False, False])

candidate_k = list(range(4, 21))
metrics_df = evaluate_k_range(X, X_reduced, candidate_k, sample_size=3000, random_state=RANDOM_STATE)
metrics_df


k= 4 | silhouette=0.0530 | calinski_harabasz=1,955 | davies_bouldin=4.3687
k= 5 | silhouette=0.0517 | calinski_harabasz=1,740 | davies_bouldin=4.2423
k= 6 | silhouette=0.0337 | calinski_harabasz=1,208 | davies_bouldin=3.9895
k= 7 | silhouette=0.0644 | calinski_harabasz=1,687 | davies_bouldin=3.8321
k= 8 | silhouette=0.0566 | calinski_harabasz=1,532 | davies_bouldin=4.1762
k= 9 | silhouette=0.0568 | calinski_harabasz=1,376 | davies_bouldin=3.9717
k=10 | silhouette=0.0636 | calinski_harabasz=1,378 | davies_bouldin=3.8360
k=11 | silhouette=0.0659 | calinski_harabasz=1,298 | davies_bouldin=3.9242
k=12 | silhouette=0.0639 | calinski_harabasz=1,253 | davies_bouldin=3.7288
k=13 | silhouette=0.0618 | calinski_harabasz=1,142 | davies_bouldin=3.9534
k=14 | silhouette=0.0606 | calinski_harabasz=1,098 | davies_bouldin=3.5396
k=15 | silhouette=0.0643 | calinski_harabasz=1,076 | davies_bouldin=3.6221
k=16 | silhouette=0.0752 | calinski_harabasz=1,038 | davies_bouldin=3.0576
k=17 | silhouette=0.0587 

,k,silhouette,calinski_harabasz,davies_bouldin,inertia,rank_silhouette,rank_ch,rank_db,rank_sum
12,16,0.075184,1037.893932,3.057554,38234.633126,1.0,13.0,1.0,15.0
3,7,0.064413,1687.174297,3.832138,41370.784955,5.0,3.0,9.0,17.0
14,18,0.069590,1002.923706,3.436401,37435.096453,2.0,14.0,3.0,19.0
16,20,0.066455,927.428746,3.320754,37150.825947,3.0,16.0,2.0,21.0
7,11,0.065916,1297.735852,3.924208,39648.797772,4.0,7.0,11.0,22.0
8,12,0.063912,1252.711504,3.728763,39210.144537,7.0,8.0,8.0,23.0
6,10,0.063558,1377.958690,3.835957,39997.154341,8.0,5.0,10.0,23.0
11,15,0.064293,1076.476864,3.622109,38484.687582,6.0,12.0,7.0,25.0
10,14,0.060618,1097.832889,3.539578,38918.992416,10.0,11.0,4.0,25.0
9,13,0.061790,1141.680326,3.953371,39254.335770,9.0,10.0,12.0,31.0


In [8]:
best_k = int(metrics_df.iloc[0]["k"])
print("Оптимальное число кластеров:", best_k)

# Финальная модель уже на полном TF-IDF пространстве
kmeans = KMeans(
    n_clusters=best_k,
    random_state=RANDOM_STATE,
    n_init=20,
    max_iter=500,
)
neg_df["cluster"] = kmeans.fit_predict(X)


Оптимальное число кластеров: 16


In [9]:
def get_top_terms_per_cluster(kmeans_model, vectorizer, top_n=12):
    feature_names = np.array(vectorizer.get_feature_names_out())
    centers = kmeans_model.cluster_centers_
    result = {}

    for cluster_id in range(centers.shape[0]):
        top_indices = centers[cluster_id].argsort()[::-1][:top_n]
        result[cluster_id] = feature_names[top_indices].tolist()

    return result

cluster_terms = get_top_terms_per_cluster(kmeans, vectorizer, top_n=12)

for cluster_id, terms in cluster_terms.items():
    print(f"Cluster {cluster_id}: {' | '.join(terms)}")


Cluster 0: аптек | лекарств | цен | препарат | фармацевт | куп | товар | покупа | покупател | рубл | прода | упаковк
Cluster 1: больниц | отделен | врач | палат | скор | больн | лежа | ребенк | род | медсестр | операц | мам
Cluster 2: кухн | рестора | обслуживан | отличн | понрав | музык | ресторан | супер | интерьер | отмеча | рожден | мест
Cluster 3: тенг | казахста | национальн | банк | енпф | рк | алмат | казахстанск | министр | экономик | фонд | компан
Cluster 4: вкусн | приятн | уютн | ед | рестора | понрав | блюд | мест | атмосфер | обслуживан | цен | ресторан
Cluster 5: анализ | врач | результат | кров | сдава | клиник | сдат | уз | гинеколог | лаборатор | результат анализ | пришл
Cluster 6: зуб | пломб | клиник | стоматолог | врач | леч | лечен | сдела | коронк | леч зуб | удал | постав
Cluster 7: отношен | пациент | врач | отношен пациент | плох отношен | отношен клиент | плох | клиент | хамск | хамск отношен | груб | больн
Cluster 8: номер | отел | завтрак | ноч | ван | гост

In [11]:
def get_representative_reviews(df, X, kmeans_model, cluster_id, n=5):
    cluster_indices = np.where(df["cluster"].values == cluster_id)[0]
    cluster_vectors = X[cluster_indices]
    center = kmeans_model.cluster_centers_[cluster_id].reshape(1, -1)

    distances = pairwise_distances(cluster_vectors, center, metric="euclidean").ravel()
    sorted_local_idx = np.argsort(distances)[:n]
    sorted_global_idx = cluster_indices[sorted_local_idx]

    return df.iloc[sorted_global_idx][["text", "clean_review", "cluster"]]

for cid in range(best_k):
    print(f"\n=== Cluster {cid} ===")
    print("Top terms:", ", ".join(cluster_terms[cid]))
    display(get_representative_reviews(neg_df, X, kmeans, cid, n=5))



=== Cluster 0 ===
Top terms: аптек, лекарств, цен, препарат, фармацевт, куп, товар, покупа, покупател, рубл, прода, упаковк


,text,clean_review,cluster
39468,Очень дорогая аптека\r\nЦены даже на простые л...,очень дорогая аптека цены даже на простые лека...,0
39345,"Аптека сумасшедших цен\r\nНепонятно, чем руков...",аптека сумасшедших цен непонятно чем руководст...,0
22688,Цены на лекарства\r\nЦены в данной аптеке очен...,цены на лекарства цены в данной аптеке очень в...,0
36648,Выбор лекарств очень скудный\r\nЦены в этой ап...,выбор лекарств очень скудный цены в этой аптек...,0
34985,Очень странная аптека\r\nОчень странная аптека...,очень странная аптека очень странная аптека ле...,0



=== Cluster 1 ===
Top terms: больниц, отделен, врач, палат, скор, больн, лежа, ребенк, род, медсестр, операц, мам


,text,clean_review,cluster
50283,Не советую\r\nЗдравствуйте всем. Недавно попал...,не советую здравствуйте всем недавно попала в ...,1
25009,Беда\r\nСам врач-хирург. Просто в шоке от боль...,беда сам врач хирург просто в шоке от больницы...,1
15976,Некомпетентность персонала\r\nПопали с дочкой ...,некомпетентность персонала попали с дочкой в к...,1
50957,Лучше в такую больницу не попадать\r\nВывод: м...,лучше в такую больницу не попадать вывод медпе...,1
2723,Смерть долгожданного ребенка\r\nЯ поступила в ...,смерть долгожданного ребенка я поступила в род...,1



=== Cluster 2 ===
Top terms: кухн, рестора, обслуживан, отличн, понрав, музык, ресторан, супер, интерьер, отмеча, рожден, мест


...

,text,clean_review,cluster
6417,Были с друзьями в количестве 6 человек . С од...,были с друзьями в количестве человек с одной с...,13
45661,Если когда то данный ресторан соответствовал с...,если когда то данный ресторан соответствовал с...,13
9139,"К сожалению , присоединюсь к тем , кто оставил...",к сожалению присоединюсь к тем кто оставил отр...,13
17062,Дата посещения : 01.03.2008 Время посещения : ...,дата посещения время посещения заказ столика в...,13
9341,"Пообедали с коллегой сегодня в этом "" шикарном...",пообедали с коллегой сегодня в этом шикарном з...,13



=== Cluster 14 ===
Top terms: русск, мест, работа, росс, дума, дела, очк, стран, туп, цен, люб, машин


,text,clean_review,cluster
0,Не было. Это пиздеш ТСа.\r\n,не было это пиздеш тса,14
228,олдскулы мои олдскулы\r\n,олдскулы мои олдскулы,14
52034,Глянь DAWG(a) если ты анимеблядь.,глянь если ты анимеблядь,14
52042,Монголокацап детектед.\r\n,монголокацап детектед,14
12301,"Чкаловский, 2-я краснодарская, Александровка\r\n",чкаловский я краснодарская александровка,14



=== Cluster 15 ===
Top terms: хохл, пиздец, хохол, русск, дегенерат, тред, лол, пост, говн, пидорашк, русн, путин


,text,clean_review,cluster
406,"Даже хохлы в гейропу не хотят, бабосы не те?\r\n",даже хохлы в гейропу не хотят бабосы не те,15
752,в два раза больше чем у хохлов,в два раза больше чем у хохлов,15
8644,в два раза больше чем у хохлов\r\n,в два раза больше чем у хохлов,15
5042,А потом будете над хохлами ржать\r\n,а потом будете над хохлами ржать,15
26428,Потому что сенегалец лучше хохла. Хохлы вообще...,потому что сенегалец лучше хохла хохлы вообще ...,15


In [12]:
cluster_summary = (
    neg_df["cluster"]
    .value_counts()
    .sort_index()
    .reset_index()
)
cluster_summary.columns = ["cluster", "count"]

print("Размеры кластеров:")
cluster_summary


Размеры кластеров:


,cluster,count
0,0,763
1,1,5878
2,2,3205
3,3,1197
4,4,3426
5,5,2577
6,6,2215
7,7,2121
8,8,3239
9,9,4573
